In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver_transactions = spark.table("digital_banking.silver.silver_transactions")

df_dim_account = spark.table("digital_banking.gold.dim_account")
df_dim_customer = spark.table("digital_banking.gold.dim_customer")
df_dim_branch = spark.table("digital_banking.gold.dim_branch")

df_enriched = df_silver_transactions \
    .join(df_dim_account.select("account_id", "customer_id", "branch_id"), 
          on="account_id", 
          how="left")

# Create fact table with measures and dimensions
df_fact_transactions = df_enriched.select(
    # Fact Primary Key
    F.col("transaction_id").alias("transaction_key"),
    
    # Foreign Keys to Dimensions
    F.col("account_id").alias("account_key"),
    F.col("customer_id").alias("customer_key"),
    F.col("branch_id").alias("branch_key"),
    
    # Date/Time Dimensions
    F.col("transaction_date"),
    F.col("transaction_timestamp"),
    F.year(F.col("transaction_date")).alias("transaction_year"),
    F.month(F.col("transaction_date")).alias("transaction_month"),
    F.dayofmonth(F.col("transaction_date")).alias("transaction_day"),
    F.dayofweek(F.col("transaction_date")).alias("transaction_day_of_week"),
    F.quarter(F.col("transaction_date")).alias("transaction_quarter"),
    F.weekofyear(F.col("transaction_date")).alias("transaction_week"),
    F.date_format(F.col("transaction_date"), "EEEE").alias("transaction_day_name"),
    F.date_format(F.col("transaction_date"), "MMMM").alias("transaction_month_name"),
    
    # Time of Day Analysis
    F.hour(F.col("transaction_timestamp")).alias("transaction_hour"),
    F.when(F.hour(F.col("transaction_timestamp")).between(6, 11), "Morning")
     .when(F.hour(F.col("transaction_timestamp")).between(12, 17), "Afternoon")
     .when(F.hour(F.col("transaction_timestamp")).between(18, 21), "Evening")
     .otherwise("Night").alias("transaction_time_of_day"),
    
    # Degenerate Dimensions (Transaction Attributes)
    F.coalesce(F.col("transaction_type"), F.lit("Unknown")).alias("transaction_type"),
    F.coalesce(F.col("transaction_channel"), F.lit("Unknown")).alias("transaction_channel"),
    F.coalesce(F.col("transaction_status"), F.lit("Unknown")).alias("transaction_status"),
    F.coalesce(F.col("currency"), F.lit("INR")).alias("currency"),
    
    # Transaction Category
    F.when(F.col("transaction_type").isin(["Deposit", "Credit"]), "Inflow")
     .when(F.col("transaction_type").isin(["Withdrawal", "Debit", "Payment"]), "Outflow")
     .when(F.col("transaction_type") == "Transfer", "Transfer")
     .otherwise("Other").alias("transaction_category"),
    
    # Channel Category
    F.when(F.col("transaction_channel").isin(["Mobile App", "Internet Banking"]), "Digital")
     .when(F.col("transaction_channel").isin(["Branch", "ATM"]), "Physical")
     .otherwise("Other").alias("channel_category"),
    
    # Merchant Information
    F.col("merchant_name"),
    F.col("merchant_category"),
    F.when(F.col("merchant_name").isNotNull(), True).otherwise(False).alias("has_merchant"),
    
    # Reference Data
    F.col("reference_number"),
    
    # MEASURES - Transaction Amount Metrics
    F.col("amount").alias("transaction_amount"),
    F.abs(F.col("amount")).alias("transaction_amount_abs"),
    
    # Amount Sign Indicator
    F.when(F.col("amount") > 0, "Positive")
     .when(F.col("amount") < 0, "Negative")
     .otherwise("Zero").alias("amount_sign"),
    
    # Amount Range Classification
    F.when(F.abs(F.col("amount")) < 1000, "< 1K")
     .when(F.abs(F.col("amount")) < 5000, "1K-5K")
     .when(F.abs(F.col("amount")) < 10000, "5K-10K")
     .when(F.abs(F.col("amount")) < 50000, "10K-50K")
     .when(F.abs(F.col("amount")) < 100000, "50K-100K")
     .otherwise("100K+").alias("amount_range"),
    
    # Status Flags
    F.when(F.col("transaction_status") == "Completed", True).otherwise(False).alias("is_completed"),
    F.when(F.col("transaction_status") == "Failed", True).otherwise(False).alias("is_failed"),
    F.when(F.col("transaction_status") == "Pending", True).otherwise(False).alias("is_pending"),
    
    # Audit Columns
    F.col("created_at").alias("source_created_at"),
    F.current_timestamp().alias("fact_created_at"),
    F.current_timestamp().alias("fact_updated_at")
)

df_fact_transactions.write \
    .mode("overwrite") \
    .partitionBy("transaction_year", "transaction_month") \
    .option("overwriteSchema", "true") \
    .saveAsTable("digital_banking.gold.fact_transactions")

print(f"Total transactions: {df_fact_transactions.count():,}")

# Display sample
display(df_fact_transactions.orderBy(F.desc("transaction_date")).limit(10))